In [ ]:
# ── Optional: run from the Zenodo deposit instead of the raw data ──────────────
# Normally every input is built from the barcode counts (Scoring.ipynb ->
# Annotations.ipynb). To run the figure notebooks from the deposited tables
# instead, uncomment the line below and run this cell once; it fills data/ and
# output/ for all six figure notebooks, so the rest of this notebook runs
# unchanged. Zenodo: https://doi.org/10.5281/zenodo.ZENODO_RECORD_ID
# All of Us data are not in the deposit (Controlled Tier; per-variant data may
# not be redistributed), so the All of Us rows of Fig. 4f and Extended Data
# Fig. 7e come out empty; every other panel reproduces.
#
# !python scripts/fetch_zenodo_data.py --record ZENODO_RECORD_ID

In [ ]:
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import copy
import numpy as np
import seaborn as sns
from scipy import stats
from scipy.stats import spearmanr, pearsonr
from scipy.stats import fisher_exact
from scipy.stats import ttest_ind
from scipy.stats import percentileofscore
from scipy.spatial import KDTree
import math
# from sklearn.metrics import r2_score
%matplotlib inline
from matplotlib.lines import Line2D
import os
import warnings
import requests
import csv
import gzip
import re
# from brokenaxes import brokenaxes
from scipy.stats import mannwhitneyu
import matplotlib.cm as cm
# import ruptures as rpt
# from joypy import joyplot
from scipy.stats import linregress
# from Bio.PDB import PDBParser, MMCIFParser, NeighborSearch, Selection
# from Bio.PDB import MMCIFParser, Polypeptide
# from Bio.PDB.MMCIF2Dict import MMCIF2Dict
from matplotlib.lines import Line2D
# from Bio import AlignIO
# from Bio import PDB
# from Bio.PDB import NeighborSearch
# from matplotlib_venn import venn3
from collections import Counter
# from Bio.PDB import PDBParser, Selection, NeighborSearch, Polypeptide
# from Bio.Data import IUPACData
from collections import Counter
# from joblib import Parallel, delayed
from tqdm import tqdm
import importlib, sys
from joypy import joyplot
import joypy.joyplot as _jp 
import matplotlib.transforms as mtransforms  
import textwrap

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 6,
    "axes.titlesize": 7,
    "axes.labelsize": 7,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "legend.fontsize": 6,
    "figure.titlesize": 7,
    "svg.fonttype": "none", #keeps text editable
    "lines.linewidth": 0.5,     # lines (e.g., axvline) stroke width
    "axes.linewidth": 0.5,      # axis spines stroke width
    "patch.linewidth": 0.5,     # bars, histograms stroke width
    # "xtick.major.width": 0.5,
    # "ytick.major.width": 0.5,
    # "xtick.minor.width": 0.5,
    # "ytick.minor.width": 0.5
})

pd.set_option('display.max_rows', 20)
pd.set_option('display.max_columns', 100)

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
path = "output/figures/fig1/"
os.makedirs(path, exist_ok=True)  

In [ ]:
from pathlib import Path
PROJECT_ROOT = Path(".").resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from labelseq_mapk.figure_data import load_scores
Replicate_scores = load_scores()

# Protein display names: MET is labelled by its protein name, HGFR, as in ED 4.
# Data lookups keep the lowercase library key.
PROT_DISPLAY_NAME = {'met': 'HGFR'}
def display_name(p):
    return PROT_DISPLAY_NAME.get(str(p).lower(), str(p).upper())
Replicate_scores = Replicate_scores.drop(columns = {'Mutation Type'})
Replicate_scores = Replicate_scores.rename(columns={'variant_category':'Mutation Type'})
Replicate_scores['Mutation Type'] = Replicate_scores['Mutation Type'].str.strip().replace('synonymous', 'synonymous wild type')
Replicate_scores['Mutation Type'] = Replicate_scores['Mutation Type'].str.strip().replace('3nt deletion', 'deletion')  
# Replicate_scores['protein'] = Replicate_scores['library'].str.replace(r'(_nterm|_cterm)$', '', regex=True)
Replicate_scores


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ── DATA PREPARATION ─────────────────────────────────────────────────────────
# Each row in Replicate_scores is already a unique (protein, variant, assay,
# assay_treatment), so no drop_duplicates needed — just recode for display.
subset = Replicate_scores[['protein', 'variant', 'assay', 'assay_treatment', 'Mutation Type']].copy()

# Merge DMSO into No_treatment
subset['assay_treatment'] = subset['assay_treatment'].replace({'DMSO': 'No_treatment'})

subset = subset[
    (subset['Mutation Type'] == 'synonymous wild type') |
    (subset['Mutation Type'] == 'missense') |
    (subset['Mutation Type'] == 'deletion') |
    (subset['Mutation Type'] == 'nonsense')
]

subset = subset[subset['Mutation Type'] != 'standard']

# Recode non-standard treatments to display label (AFTER selecting columns so
# the original row count is preserved)
subset.loc[
    (subset['assay'] == 'abundance') &
    (~subset['assay_treatment'].isin(['No_treatment', 'HSP90i'])),
    'assay_treatment'
] = 'perturb'

subset.loc[
    (subset['assay'] == 'activity') &
    (~subset['assay_treatment'].isin(['No_treatment'])),
    'assay_treatment'
] = 'perturb'

subset.loc[subset['assay'] == 'interaction', 'assay_treatment'] = 'interaction'

# Count rows per assay–treatment–protein  →  OUTER ring
counts_df = (
    subset
    .groupby(['assay', 'assay_treatment', 'protein'])
    .size()
    .reset_index(name='count')
    .sort_values(['assay', 'assay_treatment', 'protein'])
)

# Aggregate to assay × treatment  →  INNER ring
inner_df = (
    counts_df
    .groupby(['assay', 'assay_treatment'])['count']
    .sum()
    .reset_index()
    .sort_values(['assay', 'assay_treatment'])
)

print(f"Replicate_scores rows : {len(Replicate_scores):,}")
print(f"Donut total           : {counts_df['count'].sum():,}")

# ── COLOR MAPPING ─────────────────────────────────────────────────────────────
color_map = {
    ('abundance', 'No_treatment'): '#708EC0',
    ('abundance', 'HSP90i'):       '#A6C8E5',
    ('abundance', 'perturb'):      '#4C7194',
    ('activity',  'No_treatment'): '#DD8452',
    ('activity',  'perturb'):      '#F3B7A6',
}
interaction_color = '#55A868'

outer_colors = [
    interaction_color if row.assay == 'interaction'
    else color_map[(row.assay, row.assay_treatment)]
    for row in counts_df.itertuples()
]

inner_colors = [
    interaction_color if row.assay == 'interaction'
    else color_map[(row.assay, row.assay_treatment)]
    for row in inner_df.itertuples()
]

# ── FIGURE SETUP ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9.6, 8))
ax.set_aspect('equal')

outer_radius = 0.9
outer_width  = 0.27
inner_radius = 0.63
inner_width  = 0.24

# ── OUTER RING: proteins ──────────────────────────────────────────────────────
wedges_outer, _ = ax.pie(
    counts_df['count'].values,
    radius=outer_radius,
    colors=outer_colors,
    startangle=90,
    counterclock=True,
    wedgeprops={'width': outer_width, 'edgecolor': 'white', 'linewidth': 0.5},
)

# ── INNER RING: assay × treatment ────────────────────────────────────────────
wedges_inner, _ = ax.pie(
    inner_df['count'].values,
    radius=inner_radius,
    colors=inner_colors,
    startangle=90,
    counterclock=True,
    wedgeprops={'width': inner_width, 'edgecolor': 'white', 'linewidth': 1.0},
)

# ── PROTEIN LABELS — inside each outer-ring slice ────────────────────────────
r_text = outer_radius - outer_width / 2

for wedge, label in zip(wedges_outer, counts_df['protein']):
    angle_mid = (wedge.theta1 + wedge.theta2) / 2
    x = r_text * np.cos(np.radians(angle_mid))
    y = r_text * np.sin(np.radians(angle_mid))
    rotation = angle_mid if np.cos(np.radians(angle_mid)) >= 0 else angle_mid + 180
    ax.text(x, y, display_name(label),
            ha='center', va='center',
            fontsize=6.5, fontweight='bold',
            rotation=rotation, color='white')

# ── ASSAY × TREATMENT LABELS — just outside the outer ring ───────────────────
r_label = 1.17

for wedge, row in zip(wedges_inner, inner_df.itertuples()):
    angle_mid = (wedge.theta1 + wedge.theta2) / 2
    x = r_label * np.cos(np.radians(angle_mid))
    y = r_label * np.sin(np.radians(angle_mid))
    ha = 'center'
    if x > 0.15:    ha = 'left'
    elif x < -0.15: ha = 'right'
    label_text = f"{row.assay} | {row.assay_treatment}\n{row.count:,} variants"
    ax.text(x, y, label_text, ha=ha, va='center',
            fontsize=8, fontweight='bold', linespacing=1.5)

# ── LEGEND ────────────────────────────────────────────────────────────────────
legend_elements = [
    mpatches.Patch(facecolor='#4C72B0', label='Abundance – No treatment'),
    mpatches.Patch(facecolor='#A6C8E5', label='Abundance – HSP90i'),
    mpatches.Patch(facecolor='#1F4E79', label='Abundance – Perturb'),
    mpatches.Patch(facecolor='#DD8452', label='Activity – No treatment'),
    mpatches.Patch(facecolor='#F3B7A6', label='Activity – Perturb'),
    mpatches.Patch(facecolor=interaction_color, label='Interaction'),
]
ax.legend(
    handles=legend_elements,
    title="Assay × Treatment",
    loc='upper left',
    bbox_to_anchor=(1.35, 1.0),
    frameon=False,
    fontsize=9,
)

ax.set_title("Variants per Protein (outer) & Assay × Treatment (inner)", pad=15)
ax.set_xlim(-1.6, 1.6)
ax.set_ylim(-1.6, 1.6)

plt.tight_layout()
plt.savefig(path+'1d_sunburst.svg', bbox_inches='tight')


In [ ]:
# ── Publication fonts (Helvetica) ────────────────────────────────────────────
mpl.rcParams['font.family'] = 'sans-serif' 
mpl.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']  
mpl.rcParams['pdf.fonttype']    = 42
mpl.rcParams['ps.fonttype']     = 42
mpl.rcParams['svg.fonttype']    = 'none'
mpl.rcParams['savefig.dpi'] = 600

# ── Nature NPG color palette ─────────────────────────────────────────────────
npg_palette = [
    '#E64B35', '#4DBBD5', '#00A087', '#3C5488', '#F39B7F',
    '#8491B4', '#91D1C2', '#DC0000', '#7E6148', '#B09C85'
]

# Unique proteins present in DMSO / No_treatment conditions
proteins = sorted(
    Replicate_scores[
        Replicate_scores['assay_treatment'].isin(['DMSO', 'No_treatment'])
    ]['protein'].dropna().unique()
)

# Map each protein to an npg color (cycles if > 10 proteins)
protein_palette = {p: npg_palette[i % len(npg_palette)] for i, p in enumerate(proteins)}

# ── Helper: draw one replicate-correlation panel ─────────────────────────────
def draw_panel(ax, assay_name):
    subset = Replicate_scores[
        (Replicate_scores['assay'] == assay_name) &
        (Replicate_scores['assay_treatment'].isin(['DMSO', 'No_treatment'])) &
        (Replicate_scores['Mutation Type'].isin(
            ['synonymous wild type', 'missense', 'nonsense', 'deletion']  # names after the recode in cell 2; WT is not a variant
        ))
    ]
    for prot in proteins:
        sub = subset[subset['protein'] == prot]
        if sub.empty:
            continue
        ax.scatter(
            sub['corrected_score_1'], sub['corrected_score_2'],
            color=protein_palette[prot],
            alpha=0.2,
            s=6,
            rasterized=True,   # flattens dots into a bitmap in the SVG → small file
            label=prot
        )

    ax.set_xscale('log')
    ax.set_yscale('log')

    # R² computed on log10-transformed positive values
    filt = subset[(subset['corrected_score_1'] > 0) & (subset['corrected_score_2'] > 0)]
    if len(filt) > 1:
        x = np.log10(filt['corrected_score_1'])
        y = np.log10(filt['corrected_score_2'])
        _, _, r_value, _, _ = linregress(x, y)
        ax.text(0.2, 0.8, f'$r = {r_value:.2f}$',
                transform=ax.transAxes, fontsize=7,
                verticalalignment='top')

    ax.set_xlabel('WT-relative score 1', fontsize=7, fontweight='normal')
    ax.set_ylabel('WT-relative score 2', fontsize=7, fontweight='normal')
    ax.tick_params(axis='both', labelsize=7)
    ax.set_title(assay_name.capitalize(), fontsize=8)
    ax.spines[['top', 'right']].set_visible(False)

# ── Figure: abundance (top) | activity (bottom) ──────────────────────────────
# sns.set(style='white', font_scale=1)

fig, axes = plt.subplots(2, 1, figsize=(4, 4.8))

draw_panel(axes[0], 'abundance')
draw_panel(axes[1], 'activity')
# ── Legend on the right, one protein per row ─────────────────────────────────
legend_handles = [
    Line2D([0], [0], marker='o', color='w',
           markerfacecolor=protein_palette[p], markersize=6, label=display_name(p))
    for p in proteins
]
fig.legend(
    handles=legend_handles,
    loc='center left',
    bbox_to_anchor=(1.02, 0.5),
    ncol=1,
    fontsize=6,
    frameon=False,
    handletextpad=0.4,
)

plt.tight_layout(rect=[0, 0, 0.78, 1])   # leave room on the right for legend

plt.savefig(path+'1b_replicates.svg', bbox_inches='tight', dpi=300)

In [ ]:
# ── Publication fonts  ────────────────────────────────────────────
mpl.rcParams['font.family'] = 'sans-serif' 
mpl.rcParams['font.sans-serif'] = ['Arial']  
mpl.rcParams['pdf.fonttype']    = 42
mpl.rcParams['ps.fonttype']     = 42
mpl.rcParams['svg.fonttype']    = 'none'
mpl.rcParams['savefig.dpi'] = 600

# ── Nature NPG color palette ─────────────────────────────────────────────────
npg_palette = [
    '#E64B35', '#4DBBD5', '#00A087', '#3C5488', '#F39B7F',
    '#8491B4', '#91D1C2', '#DC0000', '#7E6148', '#B09C85'
]

# Unique proteins present in DMSO / No_treatment conditions
proteins = sorted(
    Replicate_scores[
        Replicate_scores['assay_treatment'].isin(['DMSO', 'No_treatment'])
    ]['protein'].dropna().unique()
)

# Map each protein to an npg color (cycles if > 10 proteins)
protein_palette = {p: npg_palette[i % len(npg_palette)] for i, p in enumerate(proteins)}

# ── Helper: draw one replicate-correlation panel ─────────────────────────────
def draw_panel(ax, assay_name):
    subset = Replicate_scores[
        (Replicate_scores['assay'] == assay_name) &
        (Replicate_scores['assay_treatment'].isin(['DMSO', 'No_treatment'])) &
        (Replicate_scores['Mutation Type'].isin(
            ['synonymous wild type', 'missense', 'nonsense', 'deletion']  # names after the recode in cell 2; WT is not a variant
        ))
    ]
    for prot in proteins:
        sub = subset[subset['protein'] == prot]
        if sub.empty:
            continue
        ax.scatter(
            sub['corrected_score_1'], sub['corrected_score_2'],
            color=protein_palette[prot],
            alpha=0.2,
            s=6,
            rasterized=True,   # flattens dots into a bitmap in the SVG → small file
            label=prot
        )

    ax.set_xscale('log')
    ax.set_yscale('log')

    # R² computed on log10-transformed positive values
    filt = subset[(subset['corrected_score_1'] > 0) & (subset['corrected_score_2'] > 0)]
    if len(filt) > 1:
        x = np.log10(filt['corrected_score_1'])
        y = np.log10(filt['corrected_score_2'])
        _, _, r_value, _, _ = linregress(x, y)
        ax.text(0.2, 0.8, f'$r = {r_value:.2f}$',
                transform=ax.transAxes, fontsize=7,
                verticalalignment='top')

    ax.set_xlabel('WT-relative score 1', fontsize=7, fontweight='normal')
    ax.set_ylabel('WT-relative score 2', fontsize=7, fontweight='normal')
    ax.tick_params(axis='both', labelsize=7)
    ax.set_title(assay_name.capitalize(), fontsize=8)
    ax.spines[['top', 'right']].set_visible(False)

# ── Figure: abundance (left) | activity (right) ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(4, 2.6))   # 2 across instead of 2 tall

draw_panel(axes[0], 'abundance')
draw_panel(axes[1], 'activity')

# ── Legend along the bottom, wrapped into rows ───────────────────────────────
legend_handles = [
    Line2D([0], [0], marker='o', color='w',
           markerfacecolor=protein_palette[p], markersize=6, label=display_name(p))
    for p in proteins
]
fig.legend(
    handles=legend_handles,
    loc='upper center',
    bbox_to_anchor=(0.5, 0.02),   # sits just below the axes
    ncol=6,                       # wrap proteins into rows; adjust to taste
    fontsize=6,
    frameon=False,
    handletextpad=0.4,
    columnspacing=1.0,
)

plt.tight_layout(rect=[0, 0.18, 1, 1])   # leave room at bottom for legend

plt.savefig(path + '1b_replicates.svg', bbox_inches='tight', dpi=300)

In [ ]:
# Your variant dictionary with types
variant_types = {
    'egfr': {
        'L858R':'common oncogenic variant',
        'G719A':'common oncogenic variant',
        'K745M':'catalytic residue or critical PTM site',
        'V948R':'critical interface',
        'I706Q':'critical interface'
    },
    'erbb2': {
        'V842I':'common oncogenic variant',
        'D769Y':'common oncogenic variant',
        'K753M':'catalytic residue or critical PTM site'
    },
    'braf': {
        'V600E':'common oncogenic variant',
        'G469A':'common oncogenic variant',
        'R509H':'critical interface',
        'R188L':'critical interface',
        'G466E':'rasopathy variant',
        'Q257R':'rasopathy variant'
    },
    'shp2': {
        'E76K':'common oncogenic variant',
        'D61Y':'common oncogenic variant',
        'N308D':'rasopathy variant',
        'C459S':'catalytic residue or critical PTM site',
        'D425K':'catalytic residue or critical PTM site',
        'R32A':'critical interface'
    },
    'kras': {
        'G12D':'common oncogenic variant',
        'Q61H':'common oncogenic variant',
        'S17N':'critical interface',
        'T35S':'critical interface',
        'C185S':'catalytic residue or critical PTM site',
        'T58I':'rasopathy variant'
    },
    'craf': {
        'R401H':'critical interface',
        'S257L':'common oncogenic variant'
    },
    'mek1': {
        'E203K':'common oncogenic variant',
        'K57N':'common oncogenic variant',
        'Y130C':'rasopathy variant',
        'Q56P':'common oncogenic variant'
    },
    'sos1': {
        'R552G':'rasopathy variant',
        'E846K':'rasopathy variant',
        'F929A':'critical interface',
        'G604V':'common oncogenic variant'
    }
}

# Map residue types to markers
type_markers = {
    'common oncogenic variant':               '*',
    'catalytic residue or critical PTM site': 's',
    'critical interface':                     '^',
    'rasopathy variant':                      'D'
}

def get_color(score):
    return '#F39B7F' if score > 1 else '#3C5488'

proteins_to_plot = ['egfr', 'erbb2', 'braf', 'shp2', 'kras', 'craf', 'mek1', 'sos1']

n_cols = 2
n_rows = len(proteins_to_plot) // n_cols   # 4 rows × 2 cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(4, 6))

for i, (ax, protein) in enumerate(zip(axes.flatten(), proteins_to_plot)):
    df_protein = Replicate_scores[
        (Replicate_scores['protein'] == protein) &
        (Replicate_scores['assay'] == 'activity') &
        ((Replicate_scores['assay_treatment'] == 'DMSO') | (Replicate_scores['assay_treatment'] == 'No_treatment'))
    ].copy()

    # Order all variants by score, low -> high (left -> right)
    df_protein = df_protein.sort_values('average score').reset_index(drop=True)
    df_protein['rank'] = range(1, len(df_protein) + 1)

    variants = variant_types[protein]

    # Reference line at score = 1 (baseline, since log axis can't show 0)
    ax.axhline(y=1, color='gray', linewidth=1, linestyle='--', zorder=1)

    # All non-highlighted variants -> grey, 50% transparent dots
    # rasterized=True: these become embedded PNG data inside the SVG,
    # keeping file size down while everything else stays true vector
    highlighted_mask = df_protein['variant'].isin(variants.keys())
    background = df_protein[~highlighted_mask]
    ax.scatter(background['rank'], background['average score'],
               color='lightgrey', marker='o', s=12, zorder=2, linewidths=0,
               alpha=0.5,
               rasterized=True)

    # Highlighted variants -> keep their type-specific color/marker (vector)
    for var, var_type in variants.items():
        row_data = df_protein[df_protein['variant'] == var]
        if not row_data.empty:
            score = row_data['average score'].values[0]
            rank = row_data['rank'].values[0]
            msize = 70 if type_markers[var_type] == '*' else 45
            ax.scatter(rank, score, marker=type_markers[var_type],
                       color=get_color(score), s=msize, zorder=3,
                       edgecolors='black', linewidths=0.4)
            ax.annotate(var, xy=(rank, score),
                        xytext=(5, 0), textcoords='offset points',
                        rotation=0, ha='left', va='center', fontsize=7)

    # Log y-axis
    ax.set_yscale('log')

    # Clean up axes
    ax.set_xticks([])
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_visible(False)

    # Tick label size (applies to remaining y-ticks)
    ax.tick_params(axis='both', labelsize=7)

    # Protein name as a title above each subplot
    ax.set_title(protein.upper(), fontsize=7, fontweight='bold')

    # Axis labels, only on outer edges
    row, col = divmod(i, n_cols)
    if row == n_rows - 1:               # only bottom row gets x-label
        ax.set_xlabel('Variants ranked by score', fontsize=7)
    if col == 0:                        # only left column gets y-label
        ax.set_ylabel('Pathway activity score', fontsize=7)

plt.tight_layout()

# dpi controls the resolution of the rasterized (grey dot) layer only;
# vector elements (highlights, text, lines) are unaffected by this setting
plt.savefig(path + '1c_waterfall.svg', bbox_inches='tight', dpi=300)

In [ ]:
import subprocess  
from pathlib import Path  
from IPython.display import Image, display  
  
RSCRIPT_BIN = "/net/gs/vol3/software/modules-sw/R/4.5.1/Linux/Ubuntu22.04/x86_64/bin/Rscript"  
  
R_SCRIPT = r'''  
.libPaths(c("~/R/library", .libPaths()))  
library(drawProteins)  
library(ggplot2)  
  
args <- commandArgs(trailingOnly = TRUE)  
out_prefix <- args[1]  

# ── Tunable layout knobs ────────────────────────────────────────────────
ROW_SPACING        <- 0.62   # fraction of the old 1.0 unit spacing between protein rows
CHAIN_HALF_HEIGHT  <- 0.11   # half-height of the grey backbone bar (was fixed at 0.20)
DOMAIN_HALF_HEIGHT <- 0.14   # half-height of the domain boxes    (was fixed at 0.25)
FIG_WIDTH          <- 10  
FIG_HEIGHT         <- 5.25   # fixed final height, no longer scales with n
  
domains <- read.delim("data/inputs/annotations/NCBI_conserved_domains_annotations/domains.txt",  
                       sep = "\t", skip = 7, stringsAsFactors = FALSE)  
domains$Query <- sub(".*>(\\w+).*", "\\1", domains$Query)  
domains$From <- as.numeric(domains$From)  
domains$To   <- as.numeric(domains$To)  
  
simplify <- function(name) {  
  if (is.na(name)) return("other")  
  name <- tolower(name)  
  if (grepl("ptk", name) || grepl("stk", name) || grepl("pk", name)) return("kinase")  
  if (grepl("sh2", name)) return("SH2")  
  if (grepl("sh3", name)) return("SH3")  
  if (grepl("gef", name)) return("GEF")  
  if (grepl("rbd", name)) return("RBD")  
  if (grepl("c1", name)) return("cysteine-rich")  
  if (grepl("tm", name) || grepl("transmembrane", name)) return("transmembrane")  
  if (grepl("lr", name)) return("leucine-rich")  
  if (grepl("sema", name)) return("extracellular")  
  if (grepl("sam", name)) return("SAM")  
  if (grepl("ras", name)) return("GTPase")  
  return(name)  
}  
domains$Label <- vapply(domains$Short.name, simplify, character(1))  
  
# Custom protein order (SHOC2 removed)  
protein_order <- c("egfr","erbb2","ret","met","grb2","shp2","sos1","sos2",  
                    "kras","mras","araf","braf","craf","ksr1","ksr2","mek1","mek2")  
protein_lengths <- c(egfr=1210, erbb2=1255, ret=1114, met=1390, grb2=217, shp2=593,  
                      sos1=1333, sos2=1319, kras=189, mras=208, araf=606, braf=766,  
                      craf=648, ksr1=921, ksr2=950, mek1=393, mek2=400)  
  
proteins <- protein_order[protein_order %in% unique(domains$Query)]  
n <- length(proteins)  
# Scaled by ROW_SPACING so rows sit closer together (first protein = top row)
order_map <- setNames(rev(seq_len(n)) * ROW_SPACING, proteins)   
  
chain_rows <- data.frame(  
  type = "CHAIN", description = toupper(proteins),  
  begin = 1, end = protein_lengths[proteins], length = protein_lengths[proteins],  
  accession = proteins, entryName = toupper(proteins), taxid = NA,  
  order = order_map[proteins], stringsAsFactors = FALSE  
)  
  
dom_sub <- domains[domains$Query %in% proteins, ]  
domain_rows <- data.frame(  
  type = "DOMAIN", description = dom_sub$Label,  
  begin = dom_sub$From, end = dom_sub$To, length = dom_sub$To - dom_sub$From,  
  accession = dom_sub$Query, entryName = toupper(dom_sub$Query), taxid = NA,  
  order = order_map[dom_sub$Query], stringsAsFactors = FALSE  
)  
  
# The BRAF-specific region (residues 41-106, N-terminal to the RBD) has no NCBI
# conserved-domain hit, so it is added from the curated domain bounds
# (config/proteins.yaml). Appended last so the other domains keep their colours.
if ("braf" %in% proteins) {
  domain_rows <- rbind(domain_rows, data.frame(
    type = "DOMAIN", description = "BRAF-specific", begin = 41, end = 106,
    length = 106 - 41, accession = "braf", entryName = "BRAF", taxid = NA,
    order = order_map[["braf"]], stringsAsFactors = FALSE))
}
plot_data <- rbind(chain_rows, domain_rows)  
  
# ── Palette: enough distinct colors that none of the domain types collide ──  
NATURE_COLORS <- c(  
  "#E64B35","#4DBBD5","#00A087","#3C5488","#F39B7F","#8491B4",  
  "#91D1C2","#DC0000","#7E6148","#B09C85","#374E55","#DF8F44",  
  "#00B4C5","#925E9F","#FDAE91","#ADB7C4",  
  "#7FC97F","#BEAED4","#FDC086","#B2882F","#386CB0","#F0027F",  
  "#BF5B17","#666666","#1B9E77","#D95F02","#7570B3","#E7298A",  
  "#66A61E","#E6AB02"  
)  
unique_labels <- unique(domain_rows$description)  
stopifnot(length(unique_labels) <= length(NATURE_COLORS))  # fail loudly, not silently wrap  
color_map <- setNames(NATURE_COLORS[seq_along(unique_labels)], unique_labels)  
  
# ── Inline labels: wide enough AND not too close to the previous label ────  
MIN_DOM_FRAC <- 0.04  
MIN_SEP_FRAC <- 0.09  
domain_rows$protein_length <- protein_lengths[domain_rows$accession]  
domain_rows$width_frac <- (domain_rows$end - domain_rows$begin) / domain_rows$protein_length  
domain_rows$mid <- (domain_rows$begin + domain_rows$end) / 2  
  
label_rows_list <- lapply(proteins, function(p) {  
  sub <- domain_rows[domain_rows$accession == p & domain_rows$width_frac >= MIN_DOM_FRAC, ]  
  sub <- sub[order(sub$mid), ]  
  min_sep <- MIN_SEP_FRAC * protein_lengths[p]  
  keep <- logical(nrow(sub))  
  last_x <- -Inf  
  for (i in seq_len(nrow(sub))) {  
    if (sub$mid[i] - last_x >= min_sep) {  
      keep[i] <- TRUE  
      last_x <- sub$mid[i]  
    }  
  }  
  sub[keep, ]  
})  
label_rows <- do.call(rbind, label_rows_list)  
  
# ── Draw manually instead of draw_chains()/draw_domains() ─────────────────
# Those two functions hardcode rect half-heights (0.2 / 0.25) with no way to
# pass a size argument, so we replicate them here with adjustable heights.
p <- draw_canvas(plot_data)  
p <- p +  
  geom_rect(data = chain_rows,  
            aes(xmin = begin, xmax = end,  
                ymin = order - CHAIN_HALF_HEIGHT, ymax = order + CHAIN_HALF_HEIGHT),  
            fill = "grey80", colour = "black", linewidth = 0.3) +  
  geom_rect(data = domain_rows,  
            aes(xmin = begin, xmax = end,  
                ymin = order - DOMAIN_HALF_HEIGHT, ymax = order + DOMAIN_HALF_HEIGHT,  
                fill = description),  
            colour = "black", linewidth = 0.2) +  
  scale_fill_manual(values = color_map, name = "Domain", guide = "none") +  
  geom_text(data = label_rows, aes(x = mid, y = order, label = description),  
            inherit.aes = FALSE, size = 2.6, color = "black") +  
  scale_y_continuous(breaks = order_map[proteins], labels = toupper(proteins)) +  
  theme_bw(base_size = 10) +  
  theme(panel.grid = element_blank(),  
        panel.border = element_blank(),   # removes the outer box  
        axis.title.y = element_blank(),  
        axis.text.y = element_text(face = "bold", size = 8)) +  
  labs(x = "Position")  
  
ggsave(paste0(out_prefix, "ED_1a_drawproteins.svg"), p, width = FIG_WIDTH, height = FIG_HEIGHT, limitsize = FALSE)  
ggsave(paste0(out_prefix, "ED_1a_drawproteins.png"), p, width = FIG_WIDTH, height = FIG_HEIGHT, dpi = 150, limitsize = FALSE)  
cat("saved\n")  
'''  
  
script_path = Path(path) / "domain_architecture_drawproteins.R"  
script_path.write_text(R_SCRIPT)  
  
result = subprocess.run(  
    [RSCRIPT_BIN, str(script_path), path],  
    capture_output=True, text=True,  
)  
if result.returncode != 0:  
    raise RuntimeError(f"R script failed:\n{result.stdout[-2000:]}\n{result.stderr[-2000:]}")  
print(result.stdout)  
  
display(Image(filename=path + "ED_1a_drawproteins.png"))

In [ ]:
import sys  
import importlib  
  
# Reload joypy to pick up a clean state before re-patching  
for mod_name in list(sys.modules.keys()):  
    if 'joypy' in mod_name:  
        importlib.reload(sys.modules[mod_name])  
  
import joypy.joyplot          # ensures 'joypy.joyplot' is in sys.modules  
from joypy import joyplot     # the plotting function you actually call  
  
_jp_mod = sys.modules['joypy.joyplot']  
_orig_flatten = _jp_mod._flatten  
_jp_mod._flatten = lambda axes: list(_orig_flatten(axes))  
  

sns.set(style='white', font_scale=0.6)

mpl.rcParams.update({
    'font.family':       'sans-serif',
    'font.sans-serif':   ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size':         6,
    'axes.labelsize':    6,
    'xtick.labelsize':   5,
    'ytick.labelsize':   5,
    'axes.linewidth':    0.5,
    'xtick.major.width': 0.5,
    'xtick.major.size':  2,
    'pdf.fonttype':      42,
    'svg.fonttype':      'none',
})

subset = Replicate_scores[
    (Replicate_scores['assay'] == 'abundance') &
    ((Replicate_scores['assay_treatment'] == 'DMSO') | (Replicate_scores['assay_treatment'] == 'No_treatment')) &
    (Replicate_scores['Mutation Type'] == 'synonymous wild type')
].copy()

# Get protein order by median score
order = (
    subset
    .groupby('protein')['intercept_0_standard-adjusted score']
    .median()
    .sort_values()
    .index
    .tolist()
)

# ALL CAPS protein labels for joyplot
order_upper = [display_name(p) for p in order]
subset['Protein'] = subset['protein'].map(display_name)
subset['Protein'] = pd.Categorical(
    subset['Protein'],
    categories=order_upper,
    ordered=True
)
subset = subset.sort_values('Protein')

fig, axes = joyplot(
    data=subset,
    by='Protein',
    column='intercept_0_standard-adjusted score',
    bins=300,
    overlap=1.0,
    fade=False,
    linecolor='black',
    linewidth=0.1,
    hist=True,
    figsize=(5, 2.5),
)

# Bold all protein name labels (joyplot adds them as text objects on each axis)
for ax in axes:
    for txt in ax.texts:
        txt.set_fontweight('bold')
        txt.set_fontsize(7)
    # Also catch ylabel if joyplot used set_ylabel
    if ax.get_ylabel():
        ax.set_ylabel(ax.get_ylabel(), fontweight='bold', fontsize=6)
    ax.set_xlim(right=8)
    ax.set_xlim(left=-0.5)

plt.xlabel('Standardized abundance score', fontsize=6, labelpad=3)

for x in [0.16, 0.53, 0.71, 1.0]:
    plt.axvline(x=x, color='gray', linestyle='dashed', linewidth=0.5, alpha=0.7)

plt.savefig(path+'ED_1b.svg', bbox_inches='tight')



In [ ]:
import sys  
import importlib  
  
# Reload joypy to pick up a clean state before re-patching  
for mod_name in list(sys.modules.keys()):  
    if 'joypy' in mod_name:  
        importlib.reload(sys.modules[mod_name])  
  
import joypy.joyplot          # ensures 'joypy.joyplot' is in sys.modules  
from joypy import joyplot     # the plotting function you actually call  
  
_jp_mod = sys.modules['joypy.joyplot']  
_orig_flatten = _jp_mod._flatten  
_jp_mod._flatten = lambda axes: list(_orig_flatten(axes))  

sns.set(style='white', font_scale=0.6)

mpl.rcParams.update({
    'font.family':       'sans-serif',
    'font.sans-serif':   ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size':         6,
    'axes.labelsize':    6,
    'xtick.labelsize':   5,
    'ytick.labelsize':   5,
    'axes.linewidth':    0.5,
    'xtick.major.width': 0.5,
    'xtick.major.size':  2,
    'pdf.fonttype':      42,
    'svg.fonttype':      'none',
})

subset = Replicate_scores[
    (Replicate_scores['assay'] == 'activity') &
    ((Replicate_scores['assay_treatment'] == 'DMSO') | (Replicate_scores['assay_treatment'] == 'No_treatment')) &
    (Replicate_scores['Mutation Type'] == 'synonymous wild type')
].copy()

# Get protein order by median score
order = (
    subset
    .groupby('protein')['intercept_0_standard-adjusted score']
    .median()
    .sort_values()
    .index
    .tolist()
)

# ALL CAPS protein labels for joyplot
order_upper = [display_name(p) for p in order]
subset['Protein'] = subset['protein'].map(display_name)
subset['Protein'] = pd.Categorical(
    subset['Protein'],
    categories=order_upper,
    ordered=True
)
subset = subset.sort_values('Protein')

fig, axes = joyplot(
    data=subset,
    by='Protein',
    column='intercept_0_standard-adjusted score',
    bins=300,
    overlap=1.0,
    fade=False,
    linecolor='black',
    linewidth=0.1,
    hist=True,
    figsize=(5, 2.5),
)

# Bold protein name labels + cap x-axis at 7
for ax in axes:
    for txt in ax.texts:
        txt.set_fontweight('bold')
        txt.set_fontsize(6)
    if ax.get_ylabel():
        ax.set_ylabel(ax.get_ylabel(), fontweight='bold', fontsize=6)
    ax.set_xlim(right=8.5)
    ax.set_xlim(left=-0.5)

plt.xlabel('Standardized activity score', fontsize=6, labelpad=3)

for x in [0.396, 1.0, 8.355, 12]:
    plt.axvline(x=x, color='gray', linestyle='dashed', linewidth=0.5, alpha=0.7)

plt.savefig(path+'ED_1c.svg', bbox_inches='tight')



In [ ]:
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, pearsonr

mpl.rcParams.update({
    'font.family':       'sans-serif',
    'font.sans-serif':   ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size':         8,
    'axes.labelsize':    8,
    'xtick.labelsize':   8,
    'ytick.labelsize':   8,
    'axes.linewidth':    0.6,
    'xtick.major.width': 0.6,
    'ytick.major.width': 0.6,
    'xtick.major.size':  3,
    'ytick.major.size':  3,
    'xtick.direction':   'out',
    'ytick.direction':   'out',
    'pdf.fonttype':      42,
    'svg.fonttype':      'none',
})

# ── Nature NPG palette ────────────────────────────────────────────────────────
NPG = [
    "#E64B35", "#4DBBD5", "#00A087", "#3C5488",
    "#F39B7F", "#8491B4", "#91D1C2", "#DC0000",
    "#7E6148", "#B09C85", "#374E55", "#DF8F44",
    "#00B4C5", "#925E9F", "#FDAE91", "#ADB7C4",
    "#2B7BB9", "#44AA99", "#DDCC77", "#AA4499",
]

# ── Pull HEK293T protein concentration from OpenCell annotations ─────────────
# opencell_curated.csv already has one row per library (e.g. 'braf_cterm',
# 'braf_nterm'), matching Replicate_scores['library'] exactly -- map
# concentration directly by library so nterm/cterm-tagged libraries of the
# same protein stay as separate points below, rather than being averaged
# together.
opencell = pd.read_csv('data/inputs/annotations/opencell_curated.csv')
conc_by_library = opencell.set_index('library')['hek_protein_conc_nm']

rep_scores_with_conc = Replicate_scores.copy()
rep_scores_with_conc['hek_protein_conc_nm'] = rep_scores_with_conc['library'].map(conc_by_library)

# ── Filter: wild type only, both assay columns present ───────────────────────
# NOTE: after the recoding in the Replicate_scores-building cell, the literal
# category for untouched wild-type clones is 'WT' (not 'wild type').
df = rep_scores_with_conc[
    (rep_scores_with_conc['Mutation Type'] == 'WT') &
    (rep_scores_with_conc['assay'] == 'abundance') &
    ((rep_scores_with_conc['assay_treatment'] == 'DMSO') | (rep_scores_with_conc['assay_treatment'] == 'No_treatment')) &
    (rep_scores_with_conc['hek_protein_conc_nm'].notna()) &
    (rep_scores_with_conc['intercept_0_standard-adjusted score'].notna())
].copy()

df['hek_protein_conc_nm'] = pd.to_numeric(df['hek_protein_conc_nm'], errors='coerce')
df = df.dropna(subset=['hek_protein_conc_nm', 'intercept_0_standard-adjusted score'])

# One point per LIBRARY (not per protein) -- e.g. braf_nterm and braf_cterm
# each plot separately, sharing braf's color. .mean() here just guards
# against any stray duplicate rows for a given library; each library
# currently has exactly one qualifying row.
plot_df = (
    df.groupby('library')
    .agg(
        protein=('protein', 'first'),
        **{
            'hek_protein_conc_nm': ('hek_protein_conc_nm', 'mean'),
            'intercept_0_standard-adjusted score': ('intercept_0_standard-adjusted score', 'mean'),
        },
    )
    .reset_index()
)

# Colors are keyed by protein so nterm/cterm libraries of the same protein
# share a color; the legend is de-duplicated to one entry per protein.
proteins  = sorted(plot_df['protein'].unique())
color_map = {p: NPG[i % len(NPG)] for i, p in enumerate(proteins)}

x = plot_df['intercept_0_standard-adjusted score'].values
y = plot_df['hek_protein_conc_nm'].values

# ── Statistics (now computed per-library rather than per-protein-mean) ──────
rho, p_spear = spearmanr(x, y)
r,   p_pear  = pearsonr(x, y)
r2 = r ** 2

# ── Figure ────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(3.25, 3.5))

for _, row in plot_df.iterrows():
    prot  = row['protein']
    color = color_map[prot]
    ax.scatter(row['intercept_0_standard-adjusted score'],
               row['hek_protein_conc_nm'],
               color=color, s=40, edgecolors='white', linewidths=0.4,
               zorder=3, label=display_name(prot))

# ── Stats annotation ──────────────────────────────────────────────────────────
stats_txt = f'$R$ = {r:.2f}\n$\\rho$ = {rho:.2f}'
ax.text(0.97, 0.97, stats_txt,
        transform=ax.transAxes,
        ha='right', va='top', fontsize=6.5,
        bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='none', alpha=0.8))

# ── Axes ──────────────────────────────────────────────────────────────────────
ax.set_xlabel('Standard abundance score',           fontsize=7, labelpad=4)
ax.set_ylabel('HEK293T protein concentration (nM)', fontsize=7, labelpad=4)

ax.set_yscale('log')
ax.spines[['top', 'right']].set_visible(False)

# ── Legend — ALL CAPS, below the plot, one entry per protein ────────────────
handles, labels = ax.get_legend_handles_labels()
seen = set()
uniq_handles, uniq_labels = [], []
for h, l in zip(handles, labels):
    if l not in seen:
        seen.add(l)
        uniq_handles.append(h)
        uniq_labels.append(l)

ax.legend(
    uniq_handles, [l.upper() for l in uniq_labels],
    fontsize=5.5,
    frameon=False,
    bbox_to_anchor=(0.5, -0.22),
    loc='upper center',
    ncol=4,
    handletextpad=0.4,
    borderpad=0,
    labelspacing=0.4,
    columnspacing=0.8,
    markerscale=0.9,
)

plt.savefig(path+'ED_1d.svg', bbox_inches='tight')
